# Prepare Data — NeMo Manifests & Ground-Truth RTTMs

This notebook:
1. Matches each audio file to its transcript PDF (via `transcripts.csv`)
2. Converts timestamps (`HH:MM:SS:CS`) to seconds
3. Generates **ground-truth RTTM** files from the transcripts
4. Generates a **NeMo manifest JSON** file ready for `run.sh`

**Naming convention:**  
`T-XXXX-L-NAME.wav` ↔ `T-XXXX-NAME-Transkr-Video.pdf`  
i.e. remove `-L-` from the audio stem.

In [1]:
import json
import wave
import pandas as pd
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE        = Path("..")                         # project root
AUDIO_DIR   = Path("D:/audio")
TRANS_CSV   = Path("transcripts.csv")
GT_RTTM_DIR = BASE / "ground_truth" / "rttm"    # output: GT RTTM files
MANIFEST_OUT = BASE / "nemo-multistage-classroom-diarization" / "manifests" / "my_data.json"

GT_RTTM_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRANS_CSV, sep=";")
print(f"Loaded {len(df)} utterances from {df['filename'].nunique()} transcripts")

Loaded 48605 utterances from 141 transcripts


In [2]:
def ts_to_sec(ts: str) -> float:
    """Convert HH:MM:SS:CS (centiseconds) to seconds."""
    h, m, s, cs = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(cs) / 100

def audio_duration(wav_path: Path) -> float:
    with wave.open(str(wav_path)) as w:
        return w.getnframes() / w.getframerate()

def audio_to_pdf_name(audio_stem: str) -> str:
    """
    T-1101-L-Lek1  ->  T-1101-Lek1-Transkr-Video.pdf
    T-1103-L-Tutor-1-1  ->  T-1103-Tutor-1-1-Transkr-Video.pdf
    """
    parts = audio_stem.split("-", 3)   # ['T', '1101', 'L', 'Lek1'] or ['T', '1101', 'L', 'Tutor-1-1']
    return f"{parts[0]}-{parts[1]}-{parts[3]}-Transkr-Video.pdf"

In [3]:
# ── Match audio files to transcripts ─────────────────────────────────────────
matched = []
unmatched = []

for wav in sorted(AUDIO_DIR.glob("*.wav")):
    pdf_name = audio_to_pdf_name(wav.stem)
    subset   = df[df["filename"] == pdf_name]
    if subset.empty:
        unmatched.append(wav.name)
    else:
        matched.append((wav, pdf_name, subset.reset_index(drop=True)))

print(f"Matched:   {len(matched)} files")
print(f"Unmatched: {len(unmatched)} files — {unmatched}")

Matched:   141 files
Unmatched: 15 files — ['T-1101-L-Lek1.wav', 'T-1101-L-Lek2.wav', 'T-1114-L-Tutor-1-4.wav', 'T-1126-L-Tutor-1-1.wav', 'T-1126-L-Tutor-1-4.wav', 'T-1208-L-Tutor-1-1.wav', 'T-1222-L-Tutor-1-4.wav', 'T-1223-L-Tutor-1-1.wav', 'T-2104-L-Tutor-1-1.wav', 'T-2104-L-Tutor-1-4.wav', 'T-2106-L-Tutor-1-1.wav', 'T-2106-L-Tutor-1-4.wav', 'T-2109-L-Tutor-1-4.wav', 'T-2111-L-Tutor-1-4.wav', 'T-2112-L-Tutor-1-1.wav']


In [4]:
# ── Generate ground-truth RTTM files ─────────────────────────────────────────
#
# RTTM format:
#   SPEAKER <file_id> 1 <start> <duration> <NA> <NA> <speaker> <NA> <NA>
#
# Duration = next_utterance_start - current_start
# For the last utterance: audio_end - last_start (capped at 10s minimum)

# Speaker codes that mean "silence / non-speech" — skip these in RTTM
NON_SPEECH = {"O"}   # O = ambient sound, not a speaker

rttm_paths = {}

for wav, pdf_name, trans in matched:
    file_id  = wav.stem                       # e.g. T-1101-L-Tutor-1-1
    dur_audio = audio_duration(wav)
    rttm_out  = GT_RTTM_DIR / f"{file_id}.rttm"
    rttm_paths[wav] = rttm_out

    # Convert all timestamps once
    starts = trans["timestamp"].apply(ts_to_sec).tolist()

    lines = []
    for i, row in trans.iterrows():
        speaker = str(row["speaker"]) if pd.notna(row["speaker"]) else ""
        if not speaker or speaker in NON_SPEECH:
            continue

        start = starts[i]
        # duration = gap until next utterance (or until audio ends)
        end   = starts[i + 1] if i + 1 < len(starts) else dur_audio
        dur   = max(0.01, end - start)          # never negative

        lines.append(
            f"SPEAKER {file_id} 1 {start:.3f} {dur:.3f} <NA> <NA> {speaker} <NA> <NA>"
        )

    rttm_out.write_text("\n".join(lines) + "\n")
    print(f"{file_id}.rttm  —  {len(lines)} segments written")

T-1101-L-Tutor-1-1.rttm  —  173 segments written
T-1101-L-Tutor-1-4.rttm  —  166 segments written
T-1103-L-Lek1.rttm  —  414 segments written
T-1103-L-Lek2.rttm  —  360 segments written
T-1103-L-Tutor-1-1.rttm  —  147 segments written
T-1103-L-Tutor-1-4.rttm  —  147 segments written
T-1104-L-Lek1.rttm  —  365 segments written
T-1104-L-Lek2.rttm  —  406 segments written
T-1104-L-Tutor-1-1.rttm  —  284 segments written
T-1104-L-Tutor-1-4.rttm  —  288 segments written


T-1106-L-Lek1.rttm  —  694 segments written


T-1106-L-Lek2.rttm  —  613 segments written
T-1106-L-Tutor-1-1.rttm  —  322 segments written
T-1106-L-Tutor-1-4.rttm  —  377 segments written
T-1107-L-Lek1.rttm  —  555 segments written
T-1107-L-Lek2.rttm  —  659 segments written
T-1107-L-Tutor-1-1.rttm  —  136 segments written
T-1107-L-Tutor-1-4.rttm  —  227 segments written
T-1109-L-Lek1.rttm  —  597 segments written


T-1109-L-Lek2.rttm  —  607 segments written
T-1109-L-Tutor-1-1.rttm  —  195 segments written


T-1109-L-Tutor-1-4.rttm  —  401 segments written
T-1110-L-Lek1.rttm  —  447 segments written
T-1110-L-Lek2.rttm  —  357 segments written
T-1110-L-Tutor-1-1.rttm  —  222 segments written
T-1110-L-Tutor-1-4.rttm  —  251 segments written
T-1113-L-Lek1.rttm  —  396 segments written
T-1113-L-Lek2.rttm  —  218 segments written
T-1113-L-Tutor-1-1.rttm  —  130 segments written
T-1113-L-Tutor-1-4.rttm  —  165 segments written
T-1114-L-Lek1.rttm  —  480 segments written


...

T-2104-L-Lek2.rttm  —  506 segments written
T-2105-L-Lek1.rttm  —  851 segments written
T-2105-L-Lek2.rttm  —  688 segments written
T-2105-L-Tutor-1-1.rttm  —  347 segments written
T-2105-L-Tutor-1-4.rttm  —  370 segments written
T-2106-L-Lek1.rttm  —  184 segments written
T-2106-L-Lek2.rttm  —  194 segments written
T-2107-L-Lek1.rttm  —  364 segments written
T-2107-L-Lek2.rttm  —  301 segments written


T-2107-L-Tutor-1-1.rttm  —  305 segments written
T-2107-L-Tutor-1-4.rttm  —  391 segments written
T-2108-L-Lek1.rttm  —  497 segments written
T-2108-L-Lek2.rttm  —  335 segments written
T-2108-L-Tutor-1-1.rttm  —  187 segments written
T-2108-L-Tutor-1-4.rttm  —  122 segments written
T-2109-L-Lek1.rttm  —  359 segments written
T-2109-L-Lek2.rttm  —  280 segments written
T-2109-L-Tutor-1-1.rttm  —  169 segments written
T-2110-L-Lek1.rttm  —  354 segments written
T-2110-L-Lek2.rttm  —  444 segments written
T-2110-L-Tutor-1-1.rttm  —  278 segments written
T-2110-L-Tutor-1-4.rttm  —  334 segments written


T-2111-L-Lek1.rttm  —  381 segments written
T-2111-L-Lek2.rttm  —  433 segments written
T-2111-L-Tutor-1-1.rttm  —  133 segments written
T-2112-L-Lek1.rttm  —  309 segments written
T-2112-L-Lek2.rttm  —  376 segments written
T-2112-L-Tutor-1-4.rttm  —  321 segments written
T-2113-L-Lek1.rttm  —  264 segments written
T-2113-L-Lek2.rttm  —  330 segments written
T-2113-L-Tutor-1-1.rttm  —  158 segments written
T-2113-L-Tutor-1-4.rttm  —  299 segments written
T-2114-L-Lek1.rttm  —  593 segments written
T-2114-L-Lek2.rttm  —  470 segments written


T-2114-L-Tutor-1-1.rttm  —  222 segments written
T-2114-L-Tutor-1-4.rttm  —  371 segments written
T-2115-L-Lek1.rttm  —  359 segments written
T-2115-L-Lek2.rttm  —  369 segments written
T-2115-L-Tutor-1-1.rttm  —  223 segments written
T-2115-L-Tutor-1-4.rttm  —  385 segments written
T-2201-L-Lek1.rttm  —  379 segments written
T-2201-L-Lek2.rttm  —  291 segments written
T-2201-L-Tutor-1-1.rttm  —  279 segments written
T-2201-L-Tutor-1-4.rttm  —  314 segments written
T-2202-L-Lek1.rttm  —  215 segments written
T-2202-L-Lek2.rttm  —  335 segments written
T-2202-L-Tutor-1-1.rttm  —  160 segments written


T-2202-L-Tutor-1-4.rttm  —  183 segments written
T-2204-L-Lek1.rttm  —  436 segments written
T-2204-L-Lek2.rttm  —  225 segments written
T-2204-L-Tutor-1-1.rttm  —  240 segments written
T-2204-L-Tutor-1-4.rttm  —  197 segments written
T-2205-L-Lek1.rttm  —  469 segments written
T-2205-L-Lek2.rttm  —  404 segments written
T-2205-L-Tutor-1-1.rttm  —  367 segments written
T-2205-L-Tutor-1-4.rttm  —  318 segments written


In [5]:
# ── Inspect the first RTTM ───────────────────────────────────────────────────
first_rttm = next(iter(rttm_paths.values()))
print(first_rttm.name)
lines = first_rttm.read_text().splitlines()
for l in lines[:10]:
    print(l)

T-1101-L-Tutor-1-1.rttm
SPEAKER T-1101-L-Tutor-1-1 1 1.060 3.210 <NA> <NA> T <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 4.270 1.920 <NA> <NA> S <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 6.190 5.980 <NA> <NA> T <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 12.170 6.970 <NA> <NA> S <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 19.140 3.890 <NA> <NA> T <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 23.030 2.240 <NA> <NA> S <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 25.270 9.930 <NA> <NA> T <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 35.200 5.880 <NA> <NA> S <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 41.080 2.170 <NA> <NA> T <NA> <NA>
SPEAKER T-1101-L-Tutor-1-1 1 43.250 9.930 <NA> <NA> T <NA> <NA>


In [6]:
# ── Count speakers per recording (for NeMo manifest num_speakers) ────────────
#
# NeMo clusters into exactly `num_speakers` groups.
# Here we count distinct speaker LABELS per recording.
# Note: 'S', 'SN', 'Ss' are all student-side roles — they get merged
# into a single 'STU' group for a teacher/student 2-class distinction.

STUDENT_CODES = {"S", "SN", "Ss", "SS", "S?", "NS", "E"}
TEACHER_CODES = {"T", "L", "LN", "LS"}

speaker_counts = {}
for wav, pdf_name, trans in matched:
    codes = set(str(s) for s in trans["speaker"].dropna().unique())
    has_teacher = bool(codes & TEACHER_CODES)
    has_student = bool(codes & STUDENT_CODES)
    n = int(has_teacher) + int(has_student)
    speaker_counts[wav] = max(n, 2)   # at least 2
    print(f"{wav.stem:35s}  codes={sorted(codes)}  num_speakers={speaker_counts[wav]}")

T-1101-L-Tutor-1-1                   codes=['S', 'T']  num_speakers=2
T-1101-L-Tutor-1-4                   codes=['S', 'T']  num_speakers=2
T-1103-L-Lek1                        codes=['E', 'O', 'S', 'T']  num_speakers=2
T-1103-L-Lek2                        codes=['S', 'T']  num_speakers=2
T-1103-L-Tutor-1-1                   codes=['S', 'T']  num_speakers=2
T-1103-L-Tutor-1-4                   codes=['S', 'T']  num_speakers=2
T-1104-L-Lek1                        codes=['E', 'S', 'T']  num_speakers=2
T-1104-L-Lek2                        codes=['E', 'O', 'S', 'T']  num_speakers=2
T-1104-L-Tutor-1-1                   codes=['S', 'T']  num_speakers=2
T-1104-L-Tutor-1-4                   codes=['NS', 'S', 'T']  num_speakers=2
T-1106-L-Lek1                        codes=['S', 'T']  num_speakers=2
T-1106-L-Lek2                        codes=['O', 'S', 'T']  num_speakers=2
T-1106-L-Tutor-1-1                   codes=['O', 'S', 'T']  num_speakers=2
T-1106-L-Tutor-1-4                   codes=['S', 

In [7]:
# ── Generate NeMo manifest JSON ───────────────────────────────────────────────
#
# One JSON object per line (JSON-Lines format).
# Paths must be absolute so NeMo can find them regardless of CWD.

manifest_lines = []

for wav, pdf_name, trans in matched:
    dur = audio_duration(wav)
    entry = {
        "audio_filepath": str(wav.resolve()),
        "offset":         0,
        "duration":       round(dur, 3),
        "label":          "infer",
        "text":           "-",
        "num_speakers":   speaker_counts[wav],
        "rttm_filepath":  str(rttm_paths[wav].resolve()),
        "uem_filepath":   None,
        "ctm_filepath":   None,
    }
    manifest_lines.append(json.dumps(entry, ensure_ascii=False))

MANIFEST_OUT.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_OUT.write_text("\n".join(manifest_lines) + "\n")
print(f"Manifest written to: {MANIFEST_OUT}")
print(f"Entries: {len(manifest_lines)}")
print()
for line in manifest_lines:
    e = json.loads(line)
    print(f"  {Path(e['audio_filepath']).name}  dur={e['duration']}s  n_spk={e['num_speakers']}")

Manifest written to: nemo-multistage-classroom-diarization\manifests\my_data.json
Entries: 141

  T-1101-L-Tutor-1-1.wav  dur=668.448s  n_spk=2
  T-1101-L-Tutor-1-4.wav  dur=943.632s  n_spk=2
  T-1103-L-Lek1.wav  dur=3190.248s  n_spk=2
  T-1103-L-Lek2.wav  dur=2881.728s  n_spk=2
  T-1103-L-Tutor-1-1.wav  dur=830.093s  n_spk=2
  T-1103-L-Tutor-1-4.wav  dur=1299.775s  n_spk=2
  T-1104-L-Lek1.wav  dur=3238.884s  n_spk=2
  T-1104-L-Lek2.wav  dur=3168.864s  n_spk=2
  T-1104-L-Tutor-1-1.wav  dur=1106.286s  n_spk=2
  T-1104-L-Tutor-1-4.wav  dur=1329.424s  n_spk=2
  T-1106-L-Lek1.wav  dur=2868.12s  n_spk=2
  T-1106-L-Lek2.wav  dur=2990.376s  n_spk=2
  T-1106-L-Tutor-1-1.wav  dur=1060.488s  n_spk=2
  T-1106-L-Tutor-1-4.wav  dur=852.228s  n_spk=2
  T-1107-L-Lek1.wav  dur=3068.532s  n_spk=2
  T-1107-L-Lek2.wav  dur=3455.28s  n_spk=2
  T-1107-L-Tutor-1-1.wav  dur=785.592s  n_spk=2
  T-1107-L-Tutor-1-4.wav  dur=814.356s  n_spk=2
  T-1109-L-Lek1.wav  dur=2474.964s  n_spk=2
  T-1109-L-Lek2.wav  dur=2

In [8]:
# ── Summary ───────────────────────────────────────────────────────────────────
total_hours = sum(audio_duration(w) for w, _, _ in matched) / 3600
total_utterances = sum(len(t) for _, _, t in matched)

print("=" * 55)
print(f"  Files ready:      {len(matched)}")
print(f"  Total audio:      {total_hours:.2f} hours")
print(f"  GT utterances:    {total_utterances}")
print(f"  GT RTTM dir:      {GT_RTTM_DIR.resolve()}")
print(f"  Manifest:         {MANIFEST_OUT.resolve()}")
print("=" * 55)
print()
print("Next step: run the diarization pipeline")
print("  cd nemo-multistage-classroom-diarization")
print("  # Edit run.sh: set MANIFEST_FILE=./manifests/my_data.json")
print("  ./run.sh")

  Files ready:      141
  Total audio:      82.02 hours
  GT utterances:    48605
  GT RTTM dir:      C:\Users\Emil\Documents\GitHub\Multi-Stage-Speaker-Diarization\ground_truth\rttm
  Manifest:         C:\Users\Emil\Documents\GitHub\Multi-Stage-Speaker-Diarization\nemo-multistage-classroom-diarization\manifests\my_data.json

Next step: run the diarization pipeline
  cd nemo-multistage-classroom-diarization
  # Edit run.sh: set MANIFEST_FILE=./manifests/my_data.json
  ./run.sh
